In [1]:
print("I am online")

I am online


In [7]:
import torch
import torch.nn as nn
import sys
sys.path.append('/mnt/data/jaitej/ronin/source')

from tcn import TemporalConvNet


class TCNTransformerNetwork(nn.Module):
    def __init__(
        self,
        input_channel=6,
        output_channel=2,
        tcn_channels=(64, 128),
        kernel_size=3,
        d_model=128,
        nhead=8,
        num_transformer_layers=4,
        dim_feedforward=512,
        dropout=0.1,
        max_len=256,
    ):
        super().__init__()

        assert tcn_channels[-1] == d_model

        # -----------------------------
        # TCN Backbone
        # -----------------------------
        self.tcn = TemporalConvNet(
            input_channel,
            list(tcn_channels),
            kernel_size,
            dropout
        )

        # -----------------------------
        # Positional Embedding
        # -----------------------------
        self.pos_embedding = nn.Parameter(
            torch.randn(1, max_len, d_model)
        )

        # -----------------------------
        # Transformer Encoder
        # -----------------------------
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_transformer_layers
        )

        # -----------------------------
        # Prediction Head
        # -----------------------------
        self.head = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.GELU(),

            nn.Linear(128, output_channel)
        )

    def forward(self, x):
        """
        x: [B, T, 6]
        """

        B, T, _ = x.shape

        # TCN expects [B, C, T]
        tcn_features = self.tcn(x.transpose(1, 2))

        # [B, T, 128]
        tcn_features = tcn_features.transpose(1, 2)

        # positional embedding
        x = tcn_features + self.pos_embedding[:, :T]

        # transformer
        transformer_features = self.transformer(x)

        # residual fusion
        features = tcn_features + transformer_features

        # velocity prediction
        out = self.head(features)

        return out

In [10]:
model = TCNTransformerNetwork()

params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable Params: {params:,}")

x = torch.randn(32, 200, 6)  # [batch, seq_len, features]

y = model(x)

print("Input Shape :", x.shape)
print("Output Shape:", y.shape)

/tmp/ipykernel_2445785/1982216090.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Trainable Params: 988,744
Input Shape : torch.Size([32, 200, 6])
Output Shape: torch.Size([32, 200, 2])
